# Libraries

In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import osmnx as ox
import geopandas as gpd
import pickle
from tqdm import tqdm
import os

In [ ]:
# Set home directory
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # CURA
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data' # SCARP

# Validation

## Road segment height

In this project, we extract the 

### Step 1: Sample 5 networks for each World Bank region (7 regions, 35 networks worldwide)

In [ ]:
# Read settlement cluster shapefiles
clusters_2564_dir = "/final_bnd/poly_2564_WB_regions.shp"
clusters_2564 = gpd.read_file(home_dir + clusters_2564_dir)

# Number of random network IDs to select per World Bank region
n_samples = 5

# Group by 'REGION_WB' and sample 'n_samples' random rows from each group
# The result will be a new DataFrame containing the sampled rows.
random_network_ids_per_region = clusters_2564.groupby('REGION_WB').sample(n=n_samples, random_state=2025)

# Save sampled network ids in csv file
random_network_ids_per_region.to_csv(home_dir+"/road_elevation_validation/sampled_35networks.csv")

In [66]:
# Print only the net_id and region
print("\nSampled 'net_id' for each region:")
print(random_network_ids_per_region[['REGION_WB', 'net_id']])


Sampled 'net_id' for each region:
                       REGION_WB  net_id
599          East Asia & Pacific  1679.0
1338         East Asia & Pacific  2849.0
1545         East Asia & Pacific  3225.0
2160         East Asia & Pacific  4215.0
1444         East Asia & Pacific  3021.0
2598       Europe & Central Asia   847.0
121        Europe & Central Asia  1143.0
1668       Europe & Central Asia   343.0
1805       Europe & Central Asia   368.0
1346       Europe & Central Asia   287.0
1639   Latin America & Caribbean  3366.0
894    Latin America & Caribbean  2002.0
1615   Latin America & Caribbean  3328.0
1965   Latin America & Caribbean  3962.0
2070   Latin America & Caribbean  4103.0
929   Middle East & North Africa  2062.0
452   Middle East & North Africa  1521.0
833   Middle East & North Africa  1926.0
1161  Middle East & North Africa  2550.0
932   Middle East & North Africa  2066.0
686                North America  1753.0
394                North America  1456.0
174                Nor

### Step 2: Extract OSM road networks

In [ ]:
# CSV file with sampled network IDs
random_network_ids_per_region = pd.read_csv(home_dir+"/road_elevation_validation/sampled_35networks.csv", index_col=0)

# Define coordinate reference system: WGS84 - World Geodetic System 1984 used in GPS
crs_lonlat = {'init': 'epsg:' + str(4326)}

# Read in the shapefile with the convex hulls of the settlement cluster boundaries
convex_hull_2564_shp = gpd.read_file(home_dir + '/final_bnd/poly_2564_WB_regions.shp').to_crs(crs_lonlat)

# Loop through the sampled network ids and extract OSM road networks
for net_id in random_network_ids_per_region.net_id.values:
    # Select convexhull poly
    convex_hull_sel = convex_hull_2564_shp[convex_hull_2564_shp["net_id"] == net_id].iloc[0]
    G = ox.graph_from_polygon(convex_hull_sel.geometry, network_type='drive_service')
    with open(home_dir + '/road_elevation_validation/sampled_networks/g_raw_drive_conv_' + str(net_id) + '.pk', 'wb') as handle:
        pickle.dump(G, handle, protocol=2)

OPTIONAL: examine road types in raw OSM networks

In [197]:
# [OPTIONAL] This cell examines the highway classes or road types
# Initiate dataframe to store road types
net_type_df = pd.DataFrame(columns=['net_id', 'net_types'], index=range(35))
counter = 0
net_type_lst = [] # Initiate list of network types

# Examine the road types
# Function to remove list nestings in highway road classes
def removeNestings(l): 
    output = []
    for i in l: 
        if type(i) == list: 
            output.extend(removeNestings(i))  # Use extend to add flattened results
        else: 
            output.append(i)
    return output

for file in os.listdir(home_dir + '/road_elevation_validation/sampled_networks/'):
    if file.endswith('.pk'):
        G = pickle.load(open(home_dir + '/road_elevation_validation/sampled_networks/' + file, 'rb'))
        net_id = int(file.split("_")[-1].split(".")[0])
        net_type_df.at[counter, 'net_id'] = net_id
        # Highway class unique values
        highway_class_dict = nx.get_edge_attributes(G,'highway')
        highway_class_lst = []
        for item in highway_class_dict.values():
            highway_class_lst.append(item)
        net_type_df.at[counter, 'net_types'] = list(set(removeNestings(highway_class_lst)))
        counter += 1
        net_type_lst.append(list(set(removeNestings(highway_class_lst))))

Below a dataframe of network id and unique road network types is created.

In [201]:
net_type_df.head(5)

,net_id,net_types
0,1521,"[tertiary, living_street, tertiary_link, resid..."
1,2550,"[tertiary, tertiary_link, residential, trunk_l..."
2,287,"[tertiary, living_street, tertiary_link, resid..."
3,3910,"[tertiary, living_street, residential, trunk_l..."
4,3481,"[tertiary, living_street, tertiary_link, resid..."


These are the unique road network types for all 35 sampled networks

In [207]:
set(removeNestings(net_type_lst))

{'yes', 'trunk_link', 'secondary_link', 'escape', 'step', 'tertiary_link', 'trunk', 'bus_stop', 'residential', 'motorway_link', 'primary_link', 'road', 'tertiary', 'living_street', 'service', 'crossing', 'unclassified', 'motorway', 'secondary', 'primary', 'busway'}

Remove highway classes that cause data bias

In [229]:
# Identify highway classes to remove
highway_classes_to_remove = [
    'yes',
    'escape',
    'step',
    'tertiary_link',
    'bus_stop',
    'residential',
    'tertiary',
    'living_street',
    'service',
    'crossing',
    'unclassified',
    'busway'
                             ]
for file in os.listdir(home_dir + '/road_elevation_validation/sampled_networks/'):
    if file.endswith('.pk'):
        net_id = int(file.split('_')[-1].split('.')[0])
        G = pickle.load(open(home_dir + '/road_elevation_validation/sampled_networks/' + file, 'rb'))
        G_lite = G.copy()
        for i,j,data in G.edges.data():
            highway_cls = data['highway']
            if highway_cls in highway_classes_to_remove:
                G_lite.remove_edge(i, j)
        # Remove isolated nodes as a result of the edge removal
        G_lite.remove_nodes_from(list(nx.isolates(G_lite)))
        with open(home_dir + '/road_elevation_validation/sampled_networks_lite/graph_cov_lite_' + str(int(net_id)) + '.pk', 'wb') as handle:
                pickle.dump(G_lite, handle, protocol=2)


### Step 3: Attach elevation values to road network

In [ ]:
# Extract road elevation using Google API
for file in os.listdir(home_dir + '/road_elevation_validation/sampled_networks_lite/'):
    if file.endswith('.pk'):
        net_id = int(file.split('_')[-1].split('.')[0])
        G = pickle.load(open(home_dir + '/road_elevation_validation/sampled_networks_lite/' + file, 'rb'))
        # add elevation to each of the nodes, using the google elevation API, then calculate edge grades
        G = ox.elevation.add_node_elevations_google(G, api_key='your API key here')
        G = ox.add_edge_grades(G)
        # Save as graphml
        ox.io.save_graphml(G, home_dir + f'/road_elevation_validation/sampled_network_lite_elev_graphml/graph_cov_lite_elev_{net_id}.graphml')
        # Save as geopackage
        ox.io.save_graph_geopackage(G, home_dir + f"/road_elevation_validation/sampled_network_lite_elev_gpkg/graph_cov_lite_elev_{net_id}.gpkg")

## Road closures

In [3]:
houston_graph_10flood = pickle.load(open(home_dir + '/Harvey_validation/G_cov_no_r_2139_10flooded.pk', 'rb'))

/var/folders/nc/ywj4rgkn46l95_tqft45f5640000gs/T/ipykernel_19125/4091128666.py:1: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  houston_graph_10flood = pickle.load(open(home_dir + '/Harvey_validation/G_cov_no_r_2139_10flooded.pk', 'rb'))


In [ ]:
# Load the nodes and edges layers from the GeoPackage into GeoDataFrames

gdf_nodes = gpd.read_file(home_dir + '/Harvey_validation/gis/G_cov_no_r_2139.gpkg', layer='nodes')
gdf_edges = gpd.read_file(home_dir + '/Harvey_validation/gis/G_cov_no_r_2139.gpkg', layer='edges')

gdf_nodes = gdf_nodes.set_index(['osmid'])
gdf_edges = gdf_edges.set_index(['u', 'v', 'key'])

houston_graph_10flooded = ox.convert.graph_from_gdfs(gdf_nodes, gdf_edges)

In [82]:
houston_graph_10flooded = ox.convert.graph_from_gdfs(gdf_nodes, gdf_edges.head(100000))

ValueError: `gdf_edges` must be multi-indexed by `(u, v, key)`.

In [ ]:
gdf_edges.head()

osmid         highway  oneway   length  FUP_5  \
u          v          key                                                      
151367091  6009226059 0    637396581  secondary_link   False   11.990    0.0   
           6009226060 0     53501801       secondary    True  154.780    0.0   
           151718533  0     53501801       secondary    True  145.323    0.0   
151367169  5180879719 0     53501801       secondary    True  281.011    0.0   
           6009226064 0     53501801       secondary    True  186.949    0.0   
...                              ...             ...     ...      ...    ...   
8172398222 8172398223 0    878461495         service   False   22.676    0.0   
8172398223 8172398224 0    878461496         service    True   47.919    0.0   
8178880448 8178880449 0    879171014         service   False   53.945    0.0   
8186932104 8186932105 0    880133303         service   False   46.689    0.0   
8193778618 8193778621 0    880963683         service   False   58.100    0.0   

                           FUP_10  FUP_20  FUP_50  FUP_75  FUP_100  ...  \
u          v          key                                           ...   
151367091  6009226059 0      0.00    0.00    0.01    0.02     0.02  ...   
           6009226060 0      0.01    0.02    0.03    0.03     0.03  ...   
           151718533  0      0.00    0.00    0.01    0.02     0.02  ...   
151367169  5180879719 0      0.11    0.19    0.31    0.34     0.35  ...   
           6009226064 0      0.11    0.19    0.31    0.34     0.35  ...   
...                           ...     ...     ...     ...      ...  ...   
8172398222 8172398223 0      0.00    0.01    0.03    0.03     0.04  ...   
8172398223 8172398224 0      0.00    0.01    0.03    0.03     0.04  ...   
8178880448 8178880449 0      0.01    0.01    0.01    0.01     0.01  ...   
8186932104 8186932105 0      0.00    0.00    0.01    0.01     0.01  ...   
8193778618 8193778621 0      0.01    0.02    0.03    0.03     0.03  ...   

                                 service  ref  bridge  access  maxspeed  \
u          v          key                                                 
151367091  6009226059 0                                                   
           6009226060 0                                                   
           151718533  0                                                   
151367169  5180879719 0                                                   
           6009226064 0                                                   
...                                  ...  ...     ...     ...       ...   
8172398222 8172398223 0                                                   
8172398223 8172398224 0    drive-through                                  
8178880448 8178880449 0                                                   
8186932104 8186932105 0         driveway                                  
8193778618 8193778621 0         driveway                                  

                           tunnel junction width landuse  \
u          v          key                                  
151367091  6009226059 0                                    
           6009226060 0                                    
           151718533  0                                    
151367169  5180879719 0                                    
           6009226064 0                                    
...                           ...      ...   ...     ...   
8172398222 8172398223 0                                    
8172398223 8172398224 0                                    
8178880448 8178880449 0                                    
8186932104 8186932105 0                                    
8193778618 8193778621 0                                    

                                                                    geometry  
u          v          key                                                     
151367091  6009226059 0    LINESTRING (-95.15599 29.63082, -95.156 29.63093)  
           600922606

In [85]:
gdf_edges.index

MultiIndex([( 151367091, 6009226059, 0),
            ( 151367091, 6009226060, 0),
            ( 151367091,  151718533, 0),
            ( 151367169, 5180879719, 0),
            ( 151367169, 6009226064, 0),
            ( 151367237,  151367238, 0),
            ( 151367238,  151367239, 0),
            ( 151367238,  151460476, 0),
            ( 151367238, 2507499447, 0),
            ( 151367239,  151367243, 0),
            ...
            (8172381403, 8172381404, 0),
            (8172381404, 8172381405, 0),
            (8172381407, 8172381411, 0),
            (8172381407, 8172381411, 1),
            (8172398219, 8172398223, 0),
            (8172398222, 8172398223, 0),
            (8172398223, 8172398224, 0),
            (8178880448, 8178880449, 0),
            (8186932104, 8186932105, 0),
            (8193778618, 8193778621, 0)],
           names=['u', 'v', 'key'], length=398750)

In [83]:
# 1. Check for null values in key columns
print("Null values:")
print(f"u: {gdf_edges['u'].isna().sum()}")
print(f"v: {gdf_edges['v'].isna().sum()}")
print(f"key: {gdf_edges['key'].isna().sum()}\n")

Null values:


KeyError: 'u'

In [60]:
a,b = ox.convert.graph_to_gdfs(houston_graph_10flood)

In [71]:
b.index

MultiIndex([( 151367091, 6009226059, 0),
            ( 151367091, 6009226060, 0),
            ( 151367169, 5180879719, 0),
            ( 151367237,  151367238, 0),
            ( 151367238,  151367239, 0),
            ( 151367238,  151367237, 0),
            ( 151367238,  151460476, 0),
            ( 151367238, 2507499447, 0),
            ( 151367239,  151367238, 0),
            ( 151367239,  151367243, 0),
            ...
            (8194117501, 6821958160, 0),
            (8194117507, 8194117513, 0),
            (8194117513, 8194117507, 0),
            (8194150027, 8194150032, 0),
            (8194150032, 8194150027, 0),
            (8194418314,  235567524, 0),
            (8194418553, 8194418560, 0),
            (8194418560, 8194418553, 0),
            (8194507774,  235709126, 0),
            (8194507780,  235709126, 0)],
           names=['u', 'v', 'key'], length=729699)

In [70]:
gdf_edges.index

MultiIndex([( 151367091, 6009226059, 0),
            ( 151367091, 6009226060, 0),
            ( 151367091,  151718533, 0),
            ( 151367169, 5180879719, 0),
            ( 151367169, 6009226064, 0),
            ( 151367237,  151367238, 0),
            ( 151367238,  151367239, 0),
            ( 151367238,  151460476, 0),
            ( 151367238, 2507499447, 0),
            ( 151367239,  151367243, 0),
            ...
            (8172381403, 8172381404, 0),
            (8172381404, 8172381405, 0),
            (8172381407, 8172381411, 0),
            (8172381407, 8172381411, 1),
            (8172398219, 8172398223, 0),
            (8172398222, 8172398223, 0),
            (8172398223, 8172398224, 0),
            (8178880448, 8178880449, 0),
            (8186932104, 8186932105, 0),
            (8193778618, 8193778621, 0)],
           names=['u', 'v', 'key'], length=398750)

In [20]:
ox.__version__

'2.0.6'

In [8]:
# Critical inundation thresholds
gamma_lst = [0.3, 0.25, 0.2, 0.15]

# Flood return period
rp = 1000

for gamma in gamma_lst:
    # Make a copy of the graph with flood inundation values
    houston_disrupted_graph = houston_graph_10flood.copy()
    # Delete nodes with flood inundation greater than 30mm or 0.3m
    for node_id, node_data in houston_graph_10flood.nodes.data():
        if node_data['FUP_' + str(rp)] >= gamma:
            houston_disrupted_graph.remove_node(node_id)
    # Delete edges with flood inundation greater than 30mm or 0.3m
    houston_disrupted_graph2 = houston_disrupted_graph.copy()
    for start_id, end_id, edge_data in houston_disrupted_graph.edges.data():
        try:
            if edge_data['FUP_' + str(rp)] >= gamma:
                houston_disrupted_graph2.remove_edge(start_id, end_id)
        except KeyError:
            continue
    break

In [10]:
houston_disrupted_graph2.number_of_nodes()

253228

In [11]:
houston_graph_10flood.number_of_nodes()

338245

## DFO

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
import rasterio.mask
from tqdm import tqdm


def count_pixels_in_polygons(gdf, raster_path, min_val=0, max_val=999):
    with rasterio.open(raster_path) as src:
        gdf = gdf.to_crs(src.crs)
        counts = []
        for geom in tqdm(gdf.geometry):
            if geom is None or geom.is_empty:
                counts.append(0)
                continue

            try:
                out_image, out_transform = rasterio.mask.mask(src, [geom], crop=True)
            except Exception:
                counts.append(0)
                continue

            arr = out_image[0] 

        

            nodata = src.nodata
            if nodata is not None:
                arr = np.where(arr == nodata, np.nan, arr)

            mask = (arr > min_val) & (arr < max_val)
            count = np.sum(mask[~np.isnan(arr)]) 

            counts.append(int(count))

        return counts


merged_gdf = gpd.read_file("lev07_v1c_merged")
shrunk_gdf = merged_gdf.copy()
shrunk_gdf['geometry'] = shrunk_gdf['geometry'].buffer(-0.01)
shrunk_gdf = shrunk_gdf[~shrunk_gdf.is_empty & shrunk_gdf.is_valid]
sindex = shrunk_gdf.sindex

counts = count_pixels_in_polygons(shrunk_gdf, "Mozambique_fathom/Fluvial/Mozambique_FU_1in1000.tif", min_val=0, max_val=999)
shrunk_gdf['counts'] = counts
shrunk_gdf[shrunk_gdf['counts']>0].to_file("basin_count0")

## Road closure